In [19]:
from typing import List, Dict, Optional
from qdrant_client import QdrantClient, models
from vector_store import VectorStore
from embedder import Embedder

class Retriever:
    def __init__(self, vector_store: VectorStore, embedder: Embedder, score_threshold: float = 0.75):
        """
        Initialize retriever with dynamic result handling
        :param score_threshold: Minimum similarity score (0-1)
        """
        self.vector_store = vector_store
        self.embedder = embedder
        self.score_threshold = score_threshold

    def retrieve(self, query: str, score_threshold: Optional[float] = None) -> List[Dict]:
        """
        Retrieve chunks with optional dynamic threshold
        """
        threshold = score_threshold if score_threshold is not None else self.score_threshold
        processed_query = self._prepare_query(query)
        query_embedding = self.embedder.generate_embeddings([processed_query])[0]

        results = self.vector_store.client.query_points(
            collection_name=self.vector_store.collection_name,
            query=query_embedding.tolist(),
            limit=1000,
            score_threshold=threshold,
            with_payload=True,
            with_vectors=False
        )

        return self._format_results(results.points)
        # return results
    
    def file_retrieval(self, filename: str):
        return self.vector_store.get_file_chunks(filename)

    def _prepare_query(self, query: str) -> str:
        """Add BGE-specific instruction prefix"""
        return f"Represent this sentence for searching relevant passages: {query}"

    def _format_results(self, scored_points) -> List[Dict]:
        """
        Format Qdrant's ScoredPoint results into a list of dictionaries
        """
        formatted_results = []
        for point in scored_points:
            formatted_results.append({
                "document_id": point.payload["document_id"],
                "chunk_order": point.payload["chunk_order"],
                "text": point.payload["text"],
                "token_count": point.payload["token_count"],
                "chunk_hash": point.payload["chunk_hash"],
                "score": point.score
            })
        return formatted_results

In [2]:

vector_store = VectorStore(collection_name="rag_chunks")




In [20]:
embedder = Embedder()

retriever = Retriever(vector_store, embedder, 0.2)

result = retriever.retrieve("What is YouTube's history?")
# result2 = retriever.hybrid_retrieve("What is YouTube's history?")

In [21]:
print(result)

[{'document_id': 'new.txt', 'chunk_order': 1, 'text': "YouTube, founded in 2005, has become the world's largest video-sharing platform...", 'token_count': 70, 'chunk_hash': 'd59c057644df16c9d68e59e1d16a4e8a', 'score': 0.6280466158966196}, {'document_id': 'new.txt', 'chunk_order': 0, 'text': '1.1 Overview of the Enterprise:', 'token_count': 5, 'chunk_hash': 'fda52433ce6df2ced1541197d411fb52', 'score': 0.4278886766577256}]
